# Transaction Fraud & Anomaly Detection: End-to-End Walkthrough

This notebook demonstrates a hybrid anomaly and fraud detection framework combining:
1. **Unsupervised Anomaly Detection** with `IsolationForest` (identifying rare, structurally atypical behavior without relying on ground-truth labels).
2. **Supervised Gradient Boosted Trees** with `HistGradientBoostingClassifier` (optimizing for class imbalance with balanced sample weighting).
3. **Time-based evaluation & threshold selection** to prevent temporal data leakage.
4. **Human-readable risk explanations** for fraud analyst triaging.

In [ ]:
import sys
from pathlib import Path

# Ensure local src package is importable
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

from fraud_anomaly.data import generate
from fraud_anomaly.features import FEATURE_COLUMNS, build_features, features_for_single
from fraud_anomaly.model import score

print("Loaded dependencies successfully.")

## 1. Synthetic Data Generation

We generate synthetic transactions reflecting realistic retail banking patterns:
- Customer-specific median spend baselines
- Normal daytime shopping patterns vs. night-time fraud bursts
- Channel distribution (POS vs Online vs ATM)
- Cross-border indicators and risk indices

In [ ]:
df = generate(n=30000, fraud_rate=0.02, seed=7)
print(f"Generated {len(df):,} transactions.")
print(f"Fraud rate: {df['label'].mean():.2%}")
df.head()

## 2. Exploratory Data Analysis (EDA)

Let us explore how transaction characteristics differ between legitimate (`0`) and fraudulent (`1`) events.

In [ ]:
# Amount distribution by class
amount_stats = df.groupby("label")["amount"].describe().T
print("Transaction Amount Statistics by Label:")
print(amount_stats)

# Channel fraud breakdown
channel_fraud = df.groupby("channel")["label"].agg(["count", "mean"]).rename(columns={"mean": "fraud_rate"})
print("\nFraud Rate by Channel:")
print(channel_fraud)

## 3. Feature Engineering

We construct numeric, stationary features:
- `log_amount`: log1p-transformed transaction value
- `amount_to_customer_median`: current amount scaled by customer's personal median spend
- `hour_sin` / `hour_cos`: cyclical temporal encoding
- `is_night`: boolean flag for night-time hours (22:00-05:00)
- One-hot encoded transaction channels and risk indices

In [ ]:
X = build_features(df)
print(f"Feature matrix shape: {X.shape}")
X.head()

## 4. Chronological Train / Test Split

In transaction fraud detection, random splits introduce look-ahead leakage. We enforce a **chronological split**:
- First 80% of days -> Training partition
- Last 20% of days -> Test partition

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df_sorted = df.sort_values("timestamp").reset_index(drop=True)

min_t = df_sorted["timestamp"].min()
max_t = df_sorted["timestamp"].max()
cutoff = min_t + 0.80 * (max_t - min_t)

train_idx = df_sorted["timestamp"] < cutoff
test_idx = ~train_idx

X_train, y_train = X[train_idx].reset_index(drop=True), df_sorted.loc[train_idx, "label"].to_numpy()
X_test, y_test = X[test_idx].reset_index(drop=True), df_sorted.loc[test_idx, "label"].to_numpy()

print(f"Train set: {len(X_train):,} transactions (Fraud rate: {y_train.mean():.2%})")
print(f"Test set:  {len(X_test):,} transactions (Fraud rate: {y_test.mean():.2%})")

## 5. Model Training: Unsupervised & Supervised

We train both:
1. `IsolationForest` (unsupervised, learning geometric isolation in feature space)
2. `HistGradientBoostingClassifier` (supervised, with class weights balanced)

In [ ]:
# 1. Isolation Forest
iso = IsolationForest(contamination=0.02, random_state=42, n_estimators=150, n_jobs=-1)
iso.fit(X_train)

train_iso_scores = -iso.score_samples(X_train)
test_iso_scores = -iso.score_samples(X_test)

# 2. HistGradientBoosting
hgb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42, max_iter=150)
hgb.fit(X_train, y_train)

train_hgb_probs = hgb.predict_proba(X_train)[:, 1]
test_hgb_probs = hgb.predict_proba(X_test)[:, 1]

print("Models fitted successfully.")

## 6. Threshold Optimization on Train Set

We select the operating threshold strictly on the **train set** targeting precision $\ge 0.80$ to minimize analyst false alarms.

In [ ]:
def pick_threshold(y_true, scores, target_p=0.80):
    p, r, t = precision_recall_curve(y_true, scores)
    valid = np.where(p[:-1] >= target_p)[0]
    if len(valid) > 0:
        best = valid[np.argmax(r[:-1][valid])]
        return float(t[best])
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-9)
    return float(t[np.argmax(f1)])

iso_thresh = pick_threshold(y_train, train_iso_scores, target_p=0.80)
hgb_thresh = pick_threshold(y_train, train_hgb_probs, target_p=0.80)

print(f"Selected Isolation Forest threshold: {iso_thresh:.4f}")
print(f"Selected HistGradientBoosting threshold: {hgb_thresh:.4f}")

## 7. Test Set Evaluation & Metrics

In [ ]:
iso_preds = (test_iso_scores >= iso_thresh).astype(int)
hgb_preds = (test_hgb_probs >= hgb_thresh).astype(int)

metrics_data = [
    {
        "Model": "Isolation Forest (Unsupervised)",
        "ROC-AUC": roc_auc_score(y_test, test_iso_scores),
        "PR-AUC": average_precision_score(y_test, test_iso_scores),
        "Precision": precision_score(y_test, iso_preds, zero_division=0),
        "Recall": recall_score(y_test, iso_preds, zero_division=0),
        "F1": f1_score(y_test, iso_preds, zero_division=0),
    },
    {
        "Model": "HistGradientBoosting (Supervised)",
        "ROC-AUC": roc_auc_score(y_test, test_hgb_probs),
        "PR-AUC": average_precision_score(y_test, test_hgb_probs),
        "Precision": precision_score(y_test, hgb_preds, zero_division=0),
        "Recall": recall_score(y_test, hgb_preds, zero_division=0),
        "F1": f1_score(y_test, hgb_preds, zero_division=0),
    },
]
pd.DataFrame(metrics_data).set_index("Model").round(4)

## 8. Precision-Recall Curves

In [ ]:
p_hgb, r_hgb, _ = precision_recall_curve(y_test, test_hgb_probs)
p_iso, r_iso, _ = precision_recall_curve(y_test, test_iso_scores)

plt.figure(figsize=(8, 5))
plt.plot(r_hgb, p_hgb, label=f"HistGradientBoosting (PR-AUC={average_precision_score(y_test, test_hgb_probs):.3f})", lw=2, color="#2563eb")
plt.plot(r_iso, p_iso, label=f"Isolation Forest (PR-AUC={average_precision_score(y_test, test_iso_scores):.3f})", lw=2, linestyle="--", color="#dc2626")
plt.axhline(y_test.mean(), color="gray", linestyle=":", label=f"Baseline ({y_test.mean():.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves on Test Set")
plt.legend(loc="lower left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 9. Explainable Inference on Single Transactions

We package models and run inference with automatic human-readable risk indicator extraction.

In [ ]:
bundle = {
    "isolation_forest": iso,
    "hgb_classifier": hgb,
    "feature_columns": FEATURE_COLUMNS,
    "iso_score_min": float(train_iso_scores.min()),
    "iso_score_max": float(train_iso_scores.max()),
    "hgb_threshold": hgb_thresh,
    "version": "1.0.0",
}

# Legitimate daytime transaction
normal_tx = {
    "amount": 32.50,
    "hour": 13,
    "day_of_week": 1,
    "channel": "pos",
    "country_risk": 0.05,
    "merchant_risk": 0.10,
    "is_foreign": False,
    "customer_median_amount": 35.00,
}
normal_result = score(bundle, features_for_single(normal_tx))
print("--- Normal Transaction Scoring ---")
print(normal_result)

# Suspicious night-time online transaction
suspicious_tx = {
    "amount": 3800.00,
    "hour": 3,
    "day_of_week": 5,
    "channel": "online",
    "country_risk": 0.88,
    "merchant_risk": 0.82,
    "is_foreign": True,
    "customer_median_amount": 45.00,
}
suspicious_result = score(bundle, features_for_single(suspicious_tx))
print("\n--- Suspicious Transaction Scoring ---")
print(suspicious_result)